In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/Users/jonathantran/Desktop/nba_economics")
STAGING_DIR = PROJECT_ROOT / "data" / "staging"
CANONICAL_DIR = PROJECT_ROOT / "data" / "canonical"

s04 = pd.read_csv(
    STAGING_DIR / "rosters" / "s04_rosters.csv"
)

print("Shape:", s04.shape)
print("\nColumns:")
print(s04.columns.tolist())

display(s04.head())

Shape: (12810, 25)

Columns:
['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'PLAYER_HEIGHT', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT', 'COLLEGE', 'COUNTRY', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'GP', 'PTS', 'REB', 'AST', 'NET_RATING', 'OREB_PCT', 'DREB_PCT', 'USG_PCT', 'TS_PCT', 'AST_PCT', 'season', 'year']


,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,AGE,PLAYER_HEIGHT,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,COLLEGE,COUNTRY,...,REB,AST,NET_RATING,OREB_PCT,DREB_PCT,USG_PCT,TS_PCT,AST_PCT,season,year
0,920,A.C. Green,1610612748,MIA,37.0,6-9,81.0,225,Oregon State,USA,...,313,39,3.8,0.076,0.151,0.137,0.492,0.050,2000-2001,2001
1,2062,A.J. Guyton,1610612741,CHI,23.0,6-1,73.0,180,Indiana,USA,...,36,64,-11.0,0.016,0.046,0.167,0.495,0.198,2000-2001,2001
2,243,Aaron McKie,1610612755,PHI,28.0,6-5,77.0,209,Temple,USA,...,311,377,7.3,0.015,0.115,0.186,0.549,0.264,2000-2001,2001
3,1425,Aaron Williams,1610612751,NJN,29.0,6-9,81.0,225,Xavier,USA,...,590,88,-7.9,0.089,0.176,0.177,0.533,0.068,2000-2001,2001
4,228,Adam Keefe,1610612744,GSW,31.0,6-9,81.0,230,Stanford,USA,...,209,36,-10.1,0.095,0.150,0.114,0.450,0.066,2000-2001,2001


In [2]:
# 1. Number of unique players
print("Unique player IDs:", s04["PLAYER_ID"].nunique())

# 2. Missing player IDs
print("Missing player IDs:", s04["PLAYER_ID"].isna().sum())

# 3. Player IDs associated with multiple names
name_counts = (
    s04.groupby("PLAYER_ID")["PLAYER_NAME"]
    .nunique()
)

print("Player IDs with multiple names:", (name_counts > 1).sum())

# 4. Check consistency of player attributes
player_fields = [
    "COUNTRY",
    "COLLEGE",
    "DRAFT_YEAR",
    "DRAFT_ROUND",
    "DRAFT_NUMBER"
]

for col in player_fields:
    inconsistent = (
        s04.groupby("PLAYER_ID")[col]
        .nunique(dropna=True)
        .gt(1)
        .sum()
    )

    print(f"{col}: {inconsistent} inconsistent player IDs")

Unique player IDs: 2584
Missing player IDs: 0
Player IDs with multiple names: 0
COUNTRY: 0 inconsistent player IDs
COLLEGE: 0 inconsistent player IDs
DRAFT_YEAR: 0 inconsistent player IDs
DRAFT_ROUND: 0 inconsistent player IDs
DRAFT_NUMBER: 0 inconsistent player IDs


In [3]:
player_columns = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "COUNTRY",
    "COLLEGE",
    "DRAFT_YEAR",
    "DRAFT_ROUND",
    "DRAFT_NUMBER"
]

c03_players = (
    s04[player_columns]
    .drop_duplicates(subset=["PLAYER_ID"])
    .rename(columns={
        "PLAYER_ID": "player_id",
        "PLAYER_NAME": "player_name",
        "COUNTRY": "country",
        "COLLEGE": "college",
        "DRAFT_YEAR": "draft_year",
        "DRAFT_ROUND": "draft_round",
        "DRAFT_NUMBER": "draft_number"
    })
    .sort_values("player_id")
    .reset_index(drop=True)
)

# Validate primary key and row count
assert len(c03_players) == 2584
assert c03_players["player_id"].is_unique
assert c03_players["player_id"].notna().all()

print("C03 players validated:", c03_players.shape)
display(c03_players.head())

C03 players validated: (2584, 7)


,player_id,player_name,country,college,draft_year,draft_round,draft_number
0,3,Grant Long,USA,Eastern Michigan,1988,2,33
1,15,Eric Piatkowski,USA,Nebraska,1994,1,15
2,21,Greg Anthony,USA,UNLV,1991,1,12
3,26,Luc Longley,Australia,New Mexico,1991,1,7
4,28,Doug West,USA,Villanova,1989,2,38


In [4]:
c03_players.to_csv(
    "../data/canonical/players.csv",
    index=False
)

print("C03 players.csv exported.")

C03 players.csv exported.
